# Create a video slideshow from images

Turn a collection of images into an animated video with Ken Burns effects, text overlays, logos, and background music — using a fully declarative Pixeltable pipeline.

## Problem

You have a set of images and want to produce a polished video — an ad, a product reel, a social media clip. Typically this means a video editor or a complex ffmpeg scripting pipeline.

| Use case | What you need |
|----------|---------------|
| Product ad | Ken Burns pan/zoom on product shots, title cards, logo, music |
| Social reel | Animated image sequence with captions |
| Photo slideshow | Smooth transitions between photos with background audio |
| Marketing clip | Branded video from static assets, no video editor needed |

## Solution

**What's in this recipe:**

- Convert still images into animated video clips with pan effects
- Control pan direction per image from table data (no Python loops)
- Add per-image captions and a logo overlay
- Concatenate all clips and add background music

The key insight: store per-clip metadata (caption text, pan direction, logo) as **table columns**. One chained computed column expression handles the entire pipeline, and Pixeltable evaluates it per row.

### Setup

In [1]:
%pip install -qU pixeltable

Note: you may need to restart the kernel to use updated packages.


In [3]:
import pixeltable as pxt
import pixeltable.functions.math as pxtmath
from pixeltable.functions.video import concat_videos_agg, with_audio

pxt.drop_dir('slideshow_demo', force=True)
pxt.create_dir('slideshow_demo')

Connected to Pixeltable database at: postgresql+psycopg://postgres:@/pixeltable?host=/Users/pjlb/.pixeltable/pgdata
Pixeltable dashboard available at: http://localhost:22089
Created directory 'slideshow_demo'.


## Step 1: Define the data

Each row is one clip in the final video. Per-clip variation comes from table columns:

- `caption`: text overlay for each clip
- `pan_sign`: `+1.0` for pan-right, `-1.0` for pan-left
- `logo`: image to overlay in the corner

This is what makes the pipeline fully declarative — no Python loops, no conditional logic.

In [4]:
LOGO_URL = 'https://raw.githubusercontent.com/pixeltable/pixeltable/main/docs/resources/pixeltable-logo-large.png'

t = pxt.create_table(
    'slideshow_demo/clips',
    {
        'image': pxt.Image,
        'seq': pxt.Int,
        'caption': pxt.String,
        'logo': pxt.Image,
        'pan_sign': pxt.Float,
    },
)

t.insert(
    [
        {
            'image': 'https://images.unsplash.com/photo-1506744038136-46273834b3fb?w=1920&q=80',
            'seq': 0,
            'caption': 'DISCOVER NATURE',
            'logo': LOGO_URL,
            'pan_sign': 1.0,
        },
        {
            'image': 'https://images.unsplash.com/photo-1470071459604-3b5ec3a7fe05?w=1920&q=80',
            'seq': 1,
            'caption': 'WILD FORESTS',
            'logo': LOGO_URL,
            'pan_sign': -1.0,
        },
        {
            'image': 'https://images.unsplash.com/photo-1441974231531-c6227db76b6e?w=1920&q=80',
            'seq': 2,
            'caption': 'SUNLIT CANOPY',
            'logo': LOGO_URL,
            'pan_sign': 1.0,
        },
        {
            'image': 'https://images.unsplash.com/photo-1507525428034-b723cf961d3e?w=1920&q=80',
            'seq': 3,
            'caption': 'OCEAN BREEZE',
            'logo': LOGO_URL,
            'pan_sign': -1.0,
        },
        {
            'image': 'https://images.unsplash.com/photo-1519681393784-d120267933ba?w=1920&q=80',
            'seq': 4,
            'caption': 'EXPLORE MORE',
            'logo': LOGO_URL,
            'pan_sign': 1.0,
        },
    ]
)

Inserted 5 rows with 0 errors in 1.04 s (4.81 rows/s)


5 rows inserted.

## Step 2: Build the video pipeline

One computed column chains the entire transformation:

```
image → static video → resize → pan effect → resize → logo overlay → text overlay
```

The pan direction alternates per row via `scroll()`, which accepts Pixeltable expressions.
`pan()` is a convenience wrapper that only takes literal direction strings, so for per-row
control we use `scroll()` directly with a `pan_sign` column:

- `pan_sign = +1.0` → `x_speed` positive, `x_start = 0` (pans right)
- `pan_sign = -1.0` → `x_speed` negative, `x_start = pan_range` (pans left)

In [5]:
W, H, DUR, CROP = 1280, 720, 4.0, 0.25

# Base: still image → video → uniform resolution
t.add_computed_column(
    base=t.image.to_video(duration=DUR).resize(width=W, height=H)
)

# Compute pan parameters from video metadata + pan_sign column
md = t.base.get_metadata()
vid_w = md.streams[0].width
viewport_w = pxtmath.floor(vid_w * (1 - CROP)).to_int()
pan_range = vid_w - viewport_w
speed = pan_range / t.base.get_duration()
x_start = pxtmath.floor(pan_range * ((1 - t.pan_sign) / 2)).to_int()

# Full pipeline: pan → resize → logo → caption
t.add_computed_column(
    clip=t.base.scroll(
        w=viewport_w, x_speed=speed * t.pan_sign, x_start=x_start
    )
    .resize(width=W, height=H)
    .overlay_image(
        t.logo,
        scale=0.10,
        opacity=0.85,
        horizontal_align='right',
        vertical_align='top',
        horizontal_margin=15,
        vertical_margin=15,
    )
    .overlay_text(
        t.caption,
        font_size=44,
        color='white',
        horizontal_align='center',
        vertical_align='bottom',
        vertical_margin=50,
        box=True,
        box_color='black',
        box_opacity=0.5,
        box_border=[8, 16],
        start_time=0.5,
        end_time=3.0,
    )
)

Added 5 column values with 0 errors in 6.43 s (0.78 rows/s)


5 rows updated.

## Step 3: Preview individual clips

Each row now has a fully rendered video clip. Let's inspect them.

In [6]:
t.select(
    t.seq, t.caption, t.pan_sign, dur=t.clip.get_duration()
).order_by(t.seq).collect()

seq,caption,pan_sign,dur
0,DISCOVER NATURE,1.,4.
1,WILD FORESTS,-1.,4.
2,SUNLIT CANOPY,1.,4.
3,OCEAN BREEZE,-1.,4.
4,EXPLORE MORE,1.,4.


## Step 4: Concatenate into final video

`concat_videos_agg` merges all clips in `seq` order into a single video.

In [7]:
video_path = (
    t.group_by()
    .select(v=concat_videos_agg(t.seq, t.clip))
    .collect()[0]['v']
)

## Step 5: Add background music

`with_audio()` replaces (or adds) the audio track on a video. The audio is trimmed to match the video duration automatically.

In [ ]:
MUSIC_URL = 'https://raw.githubusercontent.com/pixeltable/pixeltable/main/docs/resources/sample-background-music.m4a'

final = pxt.create_table(
    'slideshow_demo/final', {'video': pxt.Video, 'music': pxt.Audio}
)
final.insert([{'video': video_path, 'music': MUSIC_URL}])
final.add_computed_column(out=with_audio(final.video, final.music))

final.select(final.out).collect()

## How it works

The entire pipeline is declarative — per-clip variation comes from **data**, not code:

| Column | Purpose | Example values |
|--------|---------|----------------|
| `image` | Source photo | Unsplash URLs |
| `seq` | Clip ordering | 0, 1, 2, 3, 4 |
| `caption` | Text overlay per clip | `'DISCOVER NATURE'` |
| `logo` | Corner logo image | Logo URL |
| `pan_sign` | Pan direction | `+1.0` (right), `-1.0` (left) |

One computed column expression handles the full transformation chain. Pixeltable evaluates it per row, pulling caption, logo, and pan direction from each row's data.

### Alternative effects

Swap the pan for other built-in effects:

```python
# Zoom instead of pan
t.add_computed_column(clip=t.base.zoom(start_scale=1.0, end_scale=1.3))

# Fade-through-black transitions (add to each clip before concat)
t.add_computed_column(clip=t.base.fade_in(duration=0.5).fade_out(duration=0.5))

# Combine: pan + fade
t.add_computed_column(
    clip=pan(t.base, direction='right')
    .resize(width=1280, height=720)
    .fade_in(duration=0.5)
    .fade_out(duration=0.5)
)
```